In [111]:
pip install pandas scikit-learn nltk

**IMPORT LIBRARIES**


In [112]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nltk.download('stopwords')
stop_words=set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


** LOAD DATASET **


In [113]:
df = pd.read_csv("Resume_Screening.csv")

# Split into two columns
df[['resume_text','category']] = df['resume_text,category'].str.split(',', n=1, expand=True)

# Drop old column
df.drop('resume_text,category', axis=1, inplace=True)

# Check
print(df.head())

                                         resume_text        category
0  python machine learning pandas numpy data anal...  Data Scientist
1  data analysis statistics python sql visualization  Data Scientist
2  machine learning scikit learn pandas numpy python  Data Scientist
3     deep learning tensorflow keras neural networks  Data Scientist
4  data cleaning preprocessing pandas numpy analysis  Data Scientist


**CHECK CATEGORIES**


In [114]:
print(df.columns)

Index(['resume_text', 'category'], dtype='object')


In [115]:
print(df['category'].value_counts())

category
Data Scientist       10
Software Engineer    10
ML Engineer          10
Business Analyst     10
Data Analyst         10
Name: count, dtype: int64


**TEXT CLEANING**

In [116]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df['cleaned_text'] = df['resume_text'].apply(clean_text)

**CONVERT TEXT TO NUMBERS(TF-IDF)**


In [117]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=1000,
                             ngram_range=(1,2),
                             stop_words='english')
X = vectorizer.fit_transform(df['cleaned_text'])
y=df['category']

**TRAIN TEST SPLIT**

In [118]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

**TRAIN MODEL**


In [119]:
from sklearn.svm import LinearSVC

model = LinearSVC()
model.fit(X_train, y_train)

LinearSVC()

**PREDICTION**


In [120]:
y_pred=model.predict(X_test)

**EVALUATION**

In [121]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=1))

Accuracy: 0.7
                   precision    recall  f1-score   support

 Business Analyst       1.00      0.67      0.80         3
     Data Analyst       1.00      0.50      0.67         2
   Data Scientist       0.00      1.00      0.00         0
      ML Engineer       1.00      0.50      0.67         2
Software Engineer       1.00      1.00      1.00         3

         accuracy                           0.70        10
        macro avg       0.80      0.73      0.63        10
     weighted avg       1.00      0.70      0.81        10



**SAVE MODEL**

In [122]:
import pickle
pickle.dump(model,open("resume_model.pk1","wb"))